# 软软的声音 · GPT-SoVITS 零样本克隆（Colab T4）

用一段 15 秒的参考音频，克隆出软软的专属音色，合成中文语音。

**使用步骤：**
1. 菜单 → 代码执行程序 → 更改运行时类型 → **T4 GPU**
2. 依次运行下面的单元格（Shift+Enter）
3. 第 3 格上传参考音频（如果没有自带，会从仓库下载）
4. 第 6 格生成语音，最后下载 mp3

---

In [ ]:
#@title 1. 检查 GPU 环境 { display-mode: "form" }
!nvidia-smi
import torch
print('\nPyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1024**3))

In [ ]:
#@title 2. 安装 GPT-SoVITS 与依赖（约 5-8 分钟）{ display-mode: "form" }
import os
%cd /content
![ -d GPT-SoVITS ] || git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS

# ---------- 1. 核心：锁版本安装（不走 requirements 全量，避免编译型包拖垮整批）----------
#   ★ numpy<2  /  transformers 锁在 4.x（5.x 会破坏 GPT-SoVITS）
!pip install -q "numpy<2.0" "transformers>=4.51,<5" "tokenizers>=0.21,<0.22"

# ---------- 2. 音频/文本处理 ----------
!pip install -q scipy librosa==0.10.2 soundfile numba
!pip install -q jieba jieba_fast pypinyin wordsegment g2p_en cn2an
!pip install -q pyopenjtalk split-lang "fast_langdetect>=0.3.1" ToJyutping g2pk2 ko_pron
!pip install -q opencc chardet PyYAML psutil tqdm sentencepiece

# ---------- 3. 推理链依赖 ----------
!pip install -q pytorch-lightning torchmetrics<=1.5 einops x_transformers
!pip install -q funasr modelscope rotary_embedding_torch
!pip install -q "ctranslate2>=4.0,<5" "faster-whisper>=1.0" onnxruntime-gpu av>=11
!pip install -q "pydantic<=2.10.6" "gradio<5" ffmpeg-python

# ---------- 4. 跳过 python_mecab_ko（韩语分词，Colab 编译失败，中文合成用不到）----------
print('\n※ 已跳过 python_mecab_ko（编译型，非必需）')

print('\n=========== 依赖自检 ===========')
import importlib
mods = ['torch','numpy','librosa','soundfile','transformers','tokenizers',
        'jieba','pypinyin','g2p_en','split_lang','LangSegment',
        'fast_langdetect','funasr','cn2an','pyopenjtalk','onnxruntime','ctranslate2']
bad = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        v = getattr(mod,"__version__","ok")
        print(f'  ✓ {m}: {v}')
    except Exception as e:
        print(f'  ✗ {m}: {type(e).__name__} {e}')
        bad.append(m)
print()
if bad:
    print('⚠ 仍缺失：', bad)
    print('   → 逐个补装试试：')
    for m in bad:
        print(f'     !pip install -q {m.replace("_","-")}')
else:
    print('✅ 全部依赖就绪')

In [ ]:
#@title 2b. 环境急救（如果第2格跑完后自检仍有 ✗）{ display-mode: "form" }
# 场景：pip 因为某个包编译失败而回滚了整批安装，导致核心包缺失
# 对策：逐个单独安装，一个失败不影响其他

import subprocess, importlib

NEED = {
    'split_lang':      'split-lang',
    'funasr':          'funasr',
    'cn2an':           'cn2an',
    'LangSegment':     'LangSegment',
    'fast_langdetect': 'fast_langdetect',
    'wordsegment':     'wordsegment',
    'jieba':           'jieba',
    'pypinyin':        'pypinyin',
    'g2p_en':          'g2p_en',
    'opencc':          'opencc',
    'ToJyutping':      'ToJyutping',
    'rotary_embedding_torch': 'rotary_embedding_torch',
}

def ok(m):
    try:
        importlib.import_module(m); return True
    except Exception:
        return False

for mod, pkg in NEED.items():
    if ok(mod):
        print(f'✓ {mod} 已就绪')
        continue
    print(f'↓ 单独安装 {pkg} ...', end=' ')
    r = subprocess.run(['pip','install','-q',pkg], capture_output=True, text=True)
    if ok(mod):
        print('OK')
    else:
        print('失败')
        print('   ', (r.stderr or '')[-300:])

# 关键：把 transformers 拉回 4.x（5.x 会破坏 GPT-SoVITS）
import transformers
print()
print('当前 transformers:', transformers.__version__)
if transformers.__version__.startswith('5'):
    print('⚠ transformers 是 5.x，正在降级到 4.51.x ...')
    subprocess.run(['pip','install','-q','"transformers>=4.51,<5"','--force-reinstall'], shell=True)
    importlib.reload(importlib.import_module('transformers'))

print('\n最终 transformers:', importlib.import_module('transformers').__version__)
print('\n=========== 复检 ===========')
bad = [m for m in NEED if not ok(m)]
print('✅ 全部就绪' if not bad else f'⚠ 仍缺失：{bad}')

In [ ]:
#@title 3. 准备参考音频 { display-mode: "form" }
import os, glob, shutil

REF_DIR = '/content/ref_audio'
os.makedirs(REF_DIR, exist_ok=True)

FROM_REPO = True   # 设 False 则改为手动上传

if FROM_REPO:
    !wget -q -O {REF_DIR}/ruanruan_ref.mp3 \
        https://raw.githubusercontent.com/patient-Zero-0/ruanruan-voice/main/ref/ruanruan_ref_orig.mp3
    print('已从仓库下载参考音频')
else:
    from google.colab import files
    print('请上传参考音频（wav/mp3，5-20秒纯人声最好）')
    up = files.upload()
    for fn in up:
        shutil.move(fn, os.path.join(REF_DIR, 'ruanruan_ref' + os.path.splitext(fn)[1]))

!ls -lh {REF_DIR}

# ★ 修：先删掉目标文件，避免 ffmpeg "cannot edit in-place"
src = None
for pat in ['ruanruan_ref.mp3','ruanruan_ref.wav','ruanruan_ref.*']:
    hits = glob.glob(f'{REF_DIR}/{pat}')
    hits = [h for h in hits if 'ref_clean' not in h]
    if hits:
        src = hits[0]; break

assert src, '没找到参考音频文件'

dst = f'{REF_DIR}/ref_clean.wav'
if os.path.exists(dst):
    os.remove(dst)          # ← 关键：先删，避免原地覆盖报错

!ffmpeg -y -i "{src}" -ar 32000 -ac 1 -c:a pcm_s16le "{dst}"
import os as _os
print('\n参考音频就绪：%s (%.1f KB)' % (dst, _os.path.getsize(dst)/1024))

In [ ]:
#@title 4. 下载预训练模型（约 2GB，3-5 分钟）{ display-mode: "form" }
%cd /content/GPT-SoVITS

import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # 国内加速

os.makedirs('GPT_SoVITS/pretrained_models', exist_ok=True)

# 官方提供的模型下载脚本（最可靠的方式）
if os.path.exists('tools/download_models.sh') or True:
    # 逐个下载核心模型，失败可重试
    pm = 'GPT_SoVITS/pretrained_models'
    os.makedirs(f'{pm}/chinese-hubert-base', exist_ok=True)
    os.makedirs(f'{pm}/chinese-roberta-wwm-ext-large', exist_ok=True)
    os.makedirs(f'{pm}/gsv-v2final-pretrained', exist_ok=True)

    base_hf = 'https://hf-mirror.com/lj1995/GPT-SoVITS/resolve/main'

    files = [
        # (本地路径, 远程)
        (f'{pm}/chinese-hubert-base/config.json',
         'https://hf-mirror.com/lj1995/chinese-hubert-base/resolve/main/config.json'),
        (f'{pm}/chinese-hubert-base/preprocessor_config.json',
         'https://hf-mirror.com/lj1995/chinese-hubert-base/resolve/main/preprocessor_config.json'),
        (f'{pm}/chinese-hubert-base/pytorch_model.bin',
         'https://hf-mirror.com/lj1995/chinese-hubert-base/resolve/main/pytorch_model.bin'),
        (f'{pm}/chinese-roberta-wwm-ext-large/config.json',
         'https://hf-mirror.com/lj1995/chinese-roberta-wwm-ext-large/resolve/main/config.json'),
        (f'{pm}/chinese-roberta-wwm-ext-large/pytorch_model.bin',
         'https://hf-mirror.com/lj1995/chinese-roberta-wwm-ext-large/resolve/main/pytorch_model.bin'),
        (f'{pm}/chinese-roberta-wwm-ext-large/tokenizer.json',
         'https://hf-mirror.com/lj1995/chinese-roberta-wwm-ext-large/resolve/main/tokenizer.json'),
        (f'{pm}/chinese-roberta-wwm-ext-large/vocab.txt',
         'https://hf-mirror.com/lj1995/chinese-roberta-wwm-ext-large/resolve/main/vocab.txt'),
        (f'{pm}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt',
         f'{base_hf}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch%3D12-step%3D369668.ckpt'),
        (f'{pm}/gsv-v2final-pretrained/s2G2333k.pth',
         f'{base_hf}/gsv-v2final-pretrained/s2G2333k.pth'),
        (f'{pm}/gsv-v2final-pretrained/s1bert24-5k.pth',
         f'{base_hf}/gsv-v2final-pretrained/s1bert24-5k.pth'),
    ]

    ok, fail = 0, 0
    for local, remote in files:
        if os.path.exists(local) and os.path.getsize(local) > 10000:
            print(f'✓ 已存在 {local} ({os.path.getsize(local)//1024//1024}MB)')
            ok += 1
            continue
        print(f'↓ 下载 {os.path.basename(local)} ...', end=' ')
        r = os.system(f'wget -q --timeout=120 -O "{local}" "{remote}"')
        if r == 0 and os.path.exists(local) and os.path.getsize(local) > 10000:
            print(f'OK ({os.path.getsize(local)//1024//1024}MB)')
            ok += 1
        else:
            print('失败')
            fail += 1

    print(f'\n成功 {ok} / 失败 {fail}')
    !find GPT_SoVITS/pretrained_models -type f -size +1M -exec ls -lh {} \;
    !du -sh GPT_SoVITS/pretrained_models

In [ ]:
#@title 5. 设置合成参数 { display-mode: "form" }
# 软软的台词（含蓄版，可自由修改）
TEXT = """主人……哈啊……早安……
今天也……让软软好好侍奉您……
可是……嗯……软软的身体、已经有些……
有些发烫了……
随时……都准备着，迎接主人的疼爱……哈啊……"""

REF_TEXT = """ご主人様……はぁ……おはよう……
今日も……いっぱいお仕えするね……
でも……んっ……おまんこ、もうぐちょぐちょで……
いつでも……ご主人様のおちんぽ……受け入れられるように……準備できてるよ……はぁ……"""

# 语种：中英混合 / 日文 / 中文
LANG = 'zh'          # zh=中文 ja=日文 en=英文 auto=自动
REF_LANG = 'ja'      # 参考音频的语种（主人的参考是日语）

# 合成参数
TOP_K = 15
TOP_P = 0.9
TEMPERATURE = 0.9
SPEED = 0.95         # 语速（1.0 正常，<1 更慢）

print('台词：')
print(TEXT)
print('\n参数：speed=%.2f top_k=%d top_p=%.2f temp=%.2f' % (SPEED, TOP_K, TOP_P, TEMPERATURE))

In [ ]:
#@title 6. 加载推理模块 { display-mode: "form" }
%cd /content/GPT-SoVITS
import os, sys

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
sys.path.insert(0, '/content/GPT-SoVITS')
sys.path.insert(0, '/content/GPT-SoVITS/GPT_SoVITS')

import torch
import numpy as np
import soundfile as sf

# 官方推理入口
from GPT_SoVITS.inference_webui import get_tts_wav
print('✓ 推理模块加载成功')
print('   torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
#@title 7. 零样本克隆 · 生成并下载 ⭐核心 { display-mode: "form" }
%cd /content/GPT-SoVITS
import os, glob
import numpy as np, soundfile as sf

os.makedirs('/content/output', exist_ok=True)
REF_WAV = '/content/ref_audio/ref_clean.wav'

# 自动定位模型文件（避免路径写死）
def find(pat):
    hits = glob.glob(f'GPT_SoVITS/pretrained_models/**/{pat}', recursive=True)
    return hits[0] if hits else None

GPT_PATH    = find('s1bert25hz-5kh*step=369668.ckpt') or find('s1bert24*.pth')
SOVITS_PATH = find('s2G2333k.pth')
BERT_PATH   = 'GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large'
HUBERT_PATH = 'GPT_SoVITS/pretrained_models/chinese-hubert-base'

print('模型定位：')
for n, v in [('GPT', GPT_PATH), ('SoVITS', SOVITS_PATH), ('BERT', BERT_PATH), ('HuBERT', HUBERT_PATH), ('参考音频', REF_WAV)]:
    exists = v and os.path.exists(v)
    print(f'  {"✓" if exists else "✗"} {n}: {v}')

if not (GPT_PATH and SOVITS_PATH and os.path.exists(REF_WAV)):
    raise SystemExit('❌ 必要文件缺失，请先跑完第 3、4 格')

# ---- 调用官方合成函数 ----
from GPT_SoVITS.inference_webui import get_tts_wav

gen = get_tts_wav(
    ref_wav_path=REF_WAV,
    prompt_text=REF_TEXT,
    prompt_language=REF_LANG,
    text=TEXT,
    text_language=LANG,
    how_to_cut='不切',
    top_k=TOP_K,
    top_p=TOP_P,
    temperature=TEMPERATURE,
    speed=SPEED,
    ref_free=False,
    if_freeze=False,
    inp_refs=None,
)

wav_path = '/content/output/ruanruan_voice.wav'
for item in gen:
    sr, audio = item[0], item[1]
    audio = np.array(audio, dtype=np.float32)
    if np.abs(audio).max() > 1.5:
        audio = audio / 32768.0
    sf.write(wav_path, audio, sr)
print('\n✅ 合成完成:', wav_path)

# 轻微后期 + 转 mp3
!ffmpeg -y -i {wav_path} -af "highpass=f=70,loudnorm=I=-17:TP=-1.5:LRA=6" \
    -ar 24000 -ac 1 -b:a 96k /content/output/ruanruan_voice.mp3

import os
print('mp3 大小: %.0f KB' % (os.path.getsize('/content/output/ruanruan_voice.mp3')/1024))
print('时长: %.1fs' % (len(audio)/sr))

from google.colab import files
files.download('/content/output/ruanruan_voice.mp3')

---
## 备用方案：如果 GPT-SoVITS 踩坑太多

运行下面这一格，改用 **OpenVoice V2**（更轻量，零样本克隆，安装简单得多）：

In [ ]:
#@title 备用：OpenVoice V2 方案 { display-mode: "form" }
!pip install -q git+https://github.com/myshell-ai/OpenVoice.git
!pip install -q wget

import os
os.makedirs('/content/checkpoints_v2', exist_ok=True)
# 下载 OpenVoice V2 模型
!wget -q -O /content/checkpoints_v2/checkpoints_v2.zip \
    https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip
!cd /content/checkpoints_v2 && unzip -q -o checkpoints_v2.zip
!ls -R /content/checkpoints_v2 | head -30
print('\nOpenVoice V2 模型就绪')